# TEST_FWD: CLIM-3D Year Mapping and FWD Diagnostics

This notebook collects the CLIM-3D Marina/local year-mapping test in one place. It first checks where the Figure-8 EP-flux correction comparison lives, then uses field fingerprints to infer the Marina-to-local CLIM-3D year mapping, and only after that compares mapped-year FWD.

Default matching feature: daily polar O3 over 60-90N in March-May. The script also supports `--primary-feature u` for daily 10 hPa, 55-75N wind fingerprints; cached U fingerprints are saved after the first run.


## Figure 8 EP-Flux Correction Comparison

Figure 8 comparison files are already generated under `Longrun/Visualization/plots/`:

- `figure8_high_ozone_wind_epfdiv_rm5_no_doubar.png`: legacy EP flux, no `DO_UBAR`, no omega correction.
- `figure8_high_ozone_wind_epfdiv_rm5_doubar_omega.png`: `DO_UBAR` wind-shear/vorticity correction plus omega correction.

<table>
<tr>
<td><b>Figure 8: no DO_UBAR / no omega</b><br><img src="../Visualization/plots/figure8_high_ozone_wind_epfdiv_rm5_no_doubar.png" width="460"></td>
<td><b>Figure 8: DO_UBAR + omega</b><br><img src="../Visualization/plots/figure8_high_ozone_wind_epfdiv_rm5_doubar_omega.png" width="460"></td>
</tr>
</table>

The same low-ozone counterpart is saved as Figure 9:

<table>
<tr>
<td><b>Figure 9: no DO_UBAR / no omega</b><br><img src="../Visualization/plots/figure9_low_ozone_wind_epfdiv_rm5_no_doubar.png" width="460"></td>
<td><b>Figure 9: DO_UBAR + omega</b><br><img src="../Visualization/plots/figure9_low_ozone_wind_epfdiv_rm5_doubar_omega.png" width="460"></td>
</tr>
</table>


## Refresh The Mapping Test

Run this block to regenerate the default O3-fingerprint mapping, FWD comparison tables, and the figures displayed below. It is intentionally a thin notebook wrapper around `fwd_clim3d_feature_mapping_test.py`, so the tested logic remains reusable from the command line.


In [ ]:
import runpy
import sys
from pathlib import Path

DATE_DIR = Path.cwd()
if DATE_DIR.name != "date_treatment":
    DATE_DIR = Path("/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment")
SCRIPT = DATE_DIR / "fwd_clim3d_feature_mapping_test.py"

old_argv = sys.argv[:]
sys.argv = [str(SCRIPT), "--primary-feature", "o3"]
try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
finally:
    sys.argv = old_argv


## Key Tables

These tables are written by the script and are the most useful numbers to quote: Marina merge-history chunks, the inferred mapping, and mapped-year FWD errors.


In [ ]:
import pandas as pd
from pathlib import Path

DATE_DIR = Path.cwd()
if DATE_DIR.name != "date_treatment":
    DATE_DIR = Path("/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment")
OUT = DATE_DIR / "clim3d_feature_mapping_test"

chunks = pd.read_csv(OUT / "marina_merge_history_chunks.csv")
mapping = pd.read_csv(OUT / "feature_matched_year_mapping.csv")
summary = pd.read_csv(OUT / "feature_matched_fwd_summary.csv")
scores = pd.read_csv(OUT / "feature_chunk_sliding_window_scores.csv")

segment_summary = (
    mapping.groupby("chunk_id")
    .agg(
        marina_years=("marina_year", lambda s: f"{int(s.min())}-{int(s.max())}"),
        local_years=("our_year", lambda s: f"{int(s.min())}-{int(s.max())}"),
        marina_run=("marina_run", "first"),
        marina_source_years=("marina_source_year", lambda s: f"{int(s.min())}-{int(s.max())}"),
        match_corr=("chunk_match_correlation", "first"),
        run_match_rate=("run_matches_history", "mean"),
        source_year_match_rate=("source_year_matches_history", "mean"),
    )
    .reset_index()
)

print("Marina merge history:")
display(chunks)
print("Inferred segment mapping:")
display(segment_summary.round(4))
print("FWD summary:")
display(summary[summary["group"].str.contains("50hpa")].round(3))


## Fingerprint Matching Strength

The O3 fingerprint is the default fast test. The U fingerprint result is also shown; it was computed from daily 10 hPa, 55-75N wind and gives the same chunk mapping.

<table>
<tr>
<td><b>O3 fingerprint top windows</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_o3_daily_fingerprint_top_windows.png" width="460"></td>
<td><b>U fingerprint top windows</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_u_fingerprint_top_windows.png" width="460"></td>
</tr>
</table>


## Matched Time Segments

This timeline is the most direct view of the merged-year problem. The upper track is Marina's 200-year merged product; the lower track is the local cleaned 216-year product. Same-colored bars and connectors show the inferred chunk correspondence.

<img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_feature_matched_segment_timeline.png" width="860">


## Mapped-Year FWD Difference

After fixing the year mapping from physical-field fingerprints, the 50 hPa FWD comparison is nearly one-to-one. The remaining mean offset is about one day, which is consistent with the different local/Marina FWD date indexing conventions rather than a wrong year mapping.

<table>
<tr>
<td><b>50 hPa FWD scatter</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_feature_matched_fwd50_scatter.png" width="430"></td>
<td><b>50 hPa absolute error by pair</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_feature_matched_fwd50_abs_error.png" width="520"></td>
</tr>
</table>


## Optional: Re-run The U-Wind Fingerprint

The U-wind fingerprint is more expensive if the cache is absent because it reads the yearly 3D U files. If `local_u10_55_75N_days1_181.npy` and `marina_u10_55_75N_days1_181.npy` already exist, this is quick.


In [ ]:
# Optional heavier check. Uncomment to regenerate U-fingerprint mapping.
# import runpy, sys
# from pathlib import Path
# DATE_DIR = Path.cwd() if Path.cwd().name == "date_treatment" else Path("/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment")
# SCRIPT = DATE_DIR / "fwd_clim3d_feature_mapping_test.py"
# old_argv = sys.argv[:]
# sys.argv = [str(SCRIPT), "--primary-feature", "u"]
# try:
#     runpy.run_path(str(SCRIPT), run_name="__main__")
# finally:
#     sys.argv = old_argv


## Output Files

- Mapping CSV: `clim3d_feature_mapping_test/feature_matched_year_mapping.csv`
- FWD pair CSV: `clim3d_feature_mapping_test/feature_matched_fwd50_by_pair.csv`
- FWD summary CSV: `clim3d_feature_mapping_test/feature_matched_fwd_summary.csv`
- Figure-8 inventory CSV: `clim3d_feature_mapping_test/figure8_epflux_variant_inventory.csv`
